# How a research run works

A walk through one run of the Ageiantic Equity Research Platform: what happens, in what
order, which parts a language model does, which parts only code may do, and where a
person has to say yes.

**Every cell below reads the live codebase.** Nothing here is a transcription that can
quietly go stale — if the workflow changes, re-run the notebook and the tables change
with it. Run it with the project's own environment:

```bash
uv run jupyter lab docs/notebooks/how-a-run-works.ipynb
```

In [ ]:
from aer.agents.registry import _DEFINITIONS
from aer.config import DEFAULT_MODEL_ROUTES, Settings
from aer.core.enums import GateKind, SourceTier
from aer.workflow.workflows.vertical_slice_v1 import build_steps

steps = build_steps()
print(f"{len(steps)} steps, {sum(1 for s in steps if s.gate)} of them gates")

---
## 1. The rule everything else follows from

> **Deterministic Python owns every number and every fact. The language model owns
> planning, interpretation, comparison, adversarial challenge and writing.**

This is not a style preference, it is the design. A discounted cash flow is forty lines
of Python with unit tests; it is not a reasoning task. Putting arithmetic into a prompt
is the single most common way systems like this produce confidently wrong numbers.

So the division runs like this:

| Only code may do this | Only the model does this |
|---|---|
| Fetching, hashing, caching, rate limits | Deciding what to research |
| Parsing filings (iXBRL, PDF, HTML) | Judging whether a source is relevant |
| **All arithmetic** — ratios, WACC, DCF, comps | *Proposing* an assumption, with reasons |
| Unit and currency normalisation | Drafting prose from already-structured facts |
| Point-in-time filtering | Attacking the finished thesis |
| Verifying that a citation really says that | |

Two consequences worth holding on to while reading the rest:

* **The model may propose a citation; only code may confirm one.** Verification re-reads
  the archived document by hash and checks the quoted words actually appear in it.
* **No figure reaches a report unless it is a stored fact or a recorded calculation.**
  A number in prose that resolves to neither is refused before anyone reads it.

---
## 2. The whole run, in one picture

Blue is deterministic code. Amber is a model call. Red is a human decision — the run
stops there and nothing further is spent until somebody answers.

```mermaid
flowchart TD
    R([Operator requests a company]) --> P[plan]
    P -.model.-> PA{{planner · Opus}}
    P --> G1[/GATE 1 · PLAN/]
    G1 --> AQ[acquire<br/>EDGAR filings + XBRL]
    AQ --> CL[classify<br/>from the SIC code]
    CL --> G2[/GATE · SECTOR/]
    G2 --> PP[propose_peers]
    PP -.model.-> PPA{{peer_proposal · Sonnet}}
    PP --> G3[/GATE · PEER SET/]
    G3 --> PR[acquire_prices<br/>+ beta regression]
    PR --> EX[extract<br/>facts + excerpts]
    EX --> G4[/GATE · UK FINANCIALS/]
    G4 --> CA[calculate<br/>ratios, statements]
    G4 --> W1{{research × 5 · Sonnet}}
    CA --> CO[comps]
    CA --> AS[propose_assumptions]
    W1 --> AS
    AS -.model.-> ASA{{assumption_proposal · Opus}}
    AS --> G5[/GATE · ASSUMPTIONS/]
    G5 --> VA[value<br/>WACC + DCF + scenarios]
    VA --> DR[draft]
    CO --> DR
    DR -.model.-> DRA{{report_writer × 16 · Opus}}
    DR --> VL[validate<br/>13 metrics + citations]
    VL --> RT[red_team]
    RT -.model.-> RTA{{red_team · Opus}}
    RT --> G6[/GATE 2 · FINAL/]
    G6 --> RN[render]
    RN --> OUT([Markdown · HTML · PDF])

    classDef det fill:#dbeafe,stroke:#1e40af,color:#1e3a8a
    classDef llm fill:#fef3c7,stroke:#b45309,color:#78350f
    classDef gate fill:#fee2e2,stroke:#b91c1c,color:#7f1d1d
    class P,AQ,CL,PP,PR,EX,CA,CO,AS,VA,DR,VL,RT,RN det
    class PA,PPA,ASA,DRA,RTA,W1 llm
    class G1,G2,G3,G4,G5,G6 gate
```

Note what the shape tells you: **the model never touches the pipeline's spine.** Each
amber box hangs off a blue one. Code calls the model, takes its structured answer,
validates it, and carries on owning the data.

### The steps as the code declares them

`build_steps()` is the single source of that order. `needs` is what a step waits for —
steps with the same prerequisites run concurrently.

In [ ]:
for index, step in enumerate(steps, 1):
    cost = f"{step.estimated_cost_gbp:.2f}" if step.estimated_cost_gbp else ""
    waits = ", ".join(sorted(step.needs)) if step.needs else "-"
    print(f"{index:>2}  {step.key:<30} {step.gate or '':<18} {cost:>5}  {waits}")

---
## 3. What each agent is for

An *agent* here is a narrow thing: one role, one output contract, one tool allowlist, one
place where all of that is declared. Capability is **data in a registry**, not something
a class can grant itself — and **a new role requires an ADR**, enforced by a test that
walks each `adr` reference to the file.

In [ ]:
for definition in _DEFINITIONS:
    route = DEFAULT_MODEL_ROUTES[definition.role]
    tools = ", ".join(sorted(definition.allowed_tools)) or "no tools"
    print(f"{definition.role}  →  {route.model} at {route.effort} effort   (ADR {definition.adr})")
    print(f"    contract : {definition.output_schema_ref.split(':')[-1]}")
    print(f"    tools    : {tools}")
    print(f"    purpose  : {definition.purpose[:96]}...")
    print()

### In plain English

| Agent | What it is asked | What stops it inventing things |
|---|---|---|
| **planner** | Which sections, which sources, which risks — before anything is fetched | Holds no tools; may not state findings; a human approves the plan at gate 1 |
| **analysis** (×5 workers) | Company, industry, macro, recent developments, technical context | Its tools are *requested in a schema and executed by code* — it never calls anything itself |
| **peer_proposal** | Comparable companies, by ticker, with a reason each | Every ticker is resolved against EDGAR's registry; one that does not resolve is refused and never fetched |
| **assumption_proposal** | Exactly two numbers no filing answers: perpetual growth, exit multiple | Its schema has two fields and no others; values outside code-set bounds are *dropped, never clamped* |
| **report_writer** (×16) | One section each, from an evidence pack code assembled | No tools; every numeral must resolve to a stored fact or recorded calculation |
| **custom_section** | A section you wrote the instructions for | Skill files are additive-only — they may add requirements, never relax them |
| **validator** | Advice where the deterministic verifier could not settle something | *Advisory only*: no verdict column is writable from its output |
| **red_team** | Attack the finished thesis from a separate context | Sees the claims and evidence index only — never the drafting conversation |

---
## 4. The gates: where a person decides

A gate is a full stop. The run pauses, nothing further is spent, and the decision is
recorded with a **hash of exactly what was shown** — so approving one thing and running
another is refused rather than silently accepted.

In [ ]:
purpose = {
    "PLAN": "Before any money or any fetching: is this the right research?",
    "SECTOR_SPECIALIST": "Is this a bank/REIT/insurer? It decides which valuation models may run.",
    "PEER_SET": "Which companies is this one measured against?"
    " A bad peer moves a median without showing itself.",
    "UK_FINANCIALS": "A filing used tags the concept map does not know —"
    " confirm before they are dropped.",
    "ASSUMPTIONS": "Every number the valuation rests on that no filing answers.",
    "BUDGET": "The run would exceed its cost ceiling. Continue, or stop?",
    "FINAL": "The finished draft, with its unsupported claims and failed metrics on show.",
}
for gate in GateKind:
    conditional = gate.value in {"SECTOR_SPECIALIST", "PEER_SET", "UK_FINANCIALS", "ASSUMPTIONS"}
    when = "conditional — fires only when it applies" if conditional else "always"
    print(f"{gate.value:<18} ({when})")
    print(f"   {purpose[gate.value]}\n")

---
## 5. How a fact becomes a sentence you can trust

This is the chain the whole platform exists to protect. Every link is code.

```mermaid
flowchart LR
    F[EDGAR / filing] -->|fetch| A[Artefact<br/>stored by sha256]
    A -->|parse| FA[(Financial fact<br/>concept · value · unit · period)]
    A -->|locate| EXC[(Extraction<br/>exact character span)]
    FA -->|arithmetic| CALC[(Calculation<br/>formula · inputs · code version)]
    CALC --> CLM[Claim in the report]
    FA --> CLM
    EXC -->|citation| CLM
    CLM -->|re-read the artefact by hash| V{Does the quote<br/>actually appear?}
    V -->|yes| OK([Published])
    V -->|no| NO([Blocked · the report does not ship])

    classDef store fill:#dbeafe,stroke:#1e40af,color:#1e3a8a
    classDef check fill:#fee2e2,stroke:#b91c1c,color:#7f1d1d
    class A,FA,EXC,CALC store
    class V,NO check
```

Two properties fall out of it:

* **Point-in-time is enforced at acquisition**, not at reading. A document published after
  your as-of date never enters the run, so no later code path can use it by forgetting to
  check.
* **Units are carried through the arithmetic.** A mismatch raises; it never coerces. That
  is what stops a whole-company figure being divided by a per-share one — an error that
  is wrong by the share count and looks entirely ordinary.

In [ ]:
for tier in SourceTier:
    print(f"tier {tier.rank}  {tier.value}")
print()
print("Evidence is ranked. A section policy can demand a primary source and refuse to")
print("rest on anything below a stated tier.")

---
## 6. The rule that catches invented numbers

Every numeral a section writes must resolve to a stored fact or a recorded calculation.
The interesting part is what the rule has learned *not* to flag — each carve-out was a
live report losing sections to something that was never a quantity at all.

In [ ]:
from aer.core.section_output import unsourced_numerals

examples = [
    "Seats on Microsoft 365 continued to expand.",
    "The 10-K was filed in March 2026.",
    "There are 3 catalysts worth watching.",
    "Revenue 365 was the headline number.",
    "Margins expanded 340 basis points.",
]
for text in examples:
    problems = unsourced_numerals({"commentary": text}, [])
    print(f"{'PASSES ' if not problems else 'REFUSED'}  {text}")

`PASSES` means *no numeral needing lineage was found* — a product name (ADR 0060), a date
or filing label (ADR 0054), a count of the prose's own nouns (ADR 0057). `REFUSED` means
the sentence asserts a quantity that no claim resolves, and the draft does not stand.

### And when a draft is refused, it is repaired rather than binned

A billed, fully cited draft used to be discarded over one offending clause or for running
long. Two salvages now narrow it instead — code removing model output, never adding to it
— and the narrowed draft must pass **full** revalidation or the refusal stands.

In [ ]:
from aer.core.section_output import trimmed_to_word_count, without_unsourced_numeral_sentences

content = {
    "commentary": "The quarter was solid. Margins expanded 340 basis points. Cash conversion held."
}
print("numeral salvage :", without_unsourced_numeral_sentences(content, []))

long = {"commentary": "One two three. Four five six. Seven eight nine."}
print("length salvage  :", trimmed_to_word_count(long, 6))

---
## 7. What comes out, and what it costs

The last step assembles one document and renders it three ways. The PDF is made from the
**archived HTML bytes**, not from the object in memory, so what freezes is provably
derived from a file anyone can fetch back by hash. Only an approved report gets one.

Alongside the report the run leaves: every artefact it fetched, every fact it stored,
every calculation with its formula and inputs, every claim with its citations, thirteen
evaluation metric rows, the red team's challenges, and a cost row per model call.

In [ ]:
from aer.eval.metrics import Metric

print("Evaluation metrics scored on every run:")
for metric in Metric:
    print(" ", metric.value)

In [ ]:
projected = sum(step.estimated_cost_gbp for step in steps)
shipped = Settings.model_fields["per_run_budget_gbp"].default
here = Settings(http_user_agent="Notebook notebook@example.invalid").per_run_budget_gbp

print(f"Projected model spend for a full run : £{projected:.2f}")
print(f"Ceiling shipped as the default       : £{shipped:.2f}")
print(f"Ceiling this machine is configured to: £{here:.2f}")
print()
print("When your ceiling sits under the projection, the BUDGET gate fires part way")
print("through and asks you. That is the mechanism working, not a misconfiguration.")

The third line reads your own `.env`, so it will differ between machines — that is why it
is printed beside the shipped default rather than instead of it.

The estimates are deliberately coarse and deliberately *pre-run*: they are what the plan
gate shows you before a penny is spent. What is enforced afterwards is not the estimate
but the actual metered spend, checked in code before each step. Drafting dominates
because it is sixteen sections of Opus writing; the five research workers together cost
less than a tenth of it.

---
## 8. What it refuses to do

Worth stating plainly, because it is the point of the design:

* It will not print a figure it cannot trace to a filing or a formula.
* It will not compare against peers nobody agreed to.
* It will not run a discounted cash flow on a bank — the sector model blocks it, rather
  than footnoting it.
* It will not publish a report whose citations do not verify against the archived bytes.
* It will not spend past its ceiling because a step looked cheap.
* And it is **not regulated investment advice** — it is a personal research tool, and
  every user-facing surface says so.